# Programming assignment — Heat diffusion (PDE)

Uses `pyutils.benchmarks.heat`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..','..')))  # -> python/
import numpy as np
import matplotlib.pyplot as plt
from pyutils import set_plot_style; set_plot_style()
from pyutils.benchmarks import heat

## Task 1 — Gluing two plates

Two 5 mm plastic sheets (total `Lx=0.01` m) are pressed between surfaces at **250 °C**; initial temperature **30 °C**. Properties: $k=0.26$, $c_p=2000$, $\rho=1300$. The adhesive at the centerline fuses at **140 °C**. Find the **fusion time**. (Insulate top/bottom → effectively 1-D in x.)

In [2]:
k, cp, rho = 0.26, 2000.0, 1300.0
alpha = k/(rho*cp)
fusion_time = None
### BEGIN SOLUTION
p = heat.HeatProblem(Lx=0.01, Ly=0.01, nx=41, ny=11, alpha=alpha,
                     T_left=250.0, T_right=250.0, T_top=None, T_bottom=None,
                     T_init=30.0)
dx=p.Lx/(p.nx-1); dy=p.Ly/(p.ny-1); dt=heat.stable_dt(alpha,dx,dy)
T = np.full((p.nx,p.ny),30.0); T[0,:]=250.0; T[-1,:]=250.0
ic=p.nx//2; t=0.0
while T[ic,p.ny//2] < 140.0 and t < 1e6:
    lap=np.zeros_like(T)
    lap[1:-1,1:-1]=((T[2:,1:-1]-2*T[1:-1,1:-1]+T[:-2,1:-1])/dx**2
                    +(T[1:-1,2:]-2*T[1:-1,1:-1]+T[1:-1,:-2])/dy**2)
    T=T+dt*alpha*lap
    T[0,:]=250.0; T[-1,:]=250.0; T[:,0]=T[:,1]; T[:,-1]=T[:,-2]
    t+=dt
fusion_time=t
### END SOLUTION
print('fusion time [s]:', round(float(fusion_time),1))
assert fusion_time is not None and fusion_time > 0

fusion time [s]: 94.8


## Task 2 — Circuit board with heat sources

Place two localized heat sources (chips) in a 2-D board with cold (Dirichlet) edges. Plot the steady temperature and report the **hot-spot temperature**. Then double the source strength and observe the change.

In [3]:
def make_source(strength):
    def S(X,Y,t):
        s=np.zeros_like(X)
        for (cx,cy) in [(0.35,0.5),(0.65,0.5)]:
            s += strength*np.exp(-((X-cx)**2+(Y-cy)**2)/(2*0.05**2))
        return s
    return S
### BEGIN SOLUTION
p = heat.HeatProblem(nx=51, ny=51, alpha=1e-4, T_left=0.0, T_right=0.0,
                     T_top=0.0, T_bottom=0.0, source=make_source(2e-2))
x,y,T = heat.solve(p, t_end=0.5*p.Lx**2/p.alpha)
hotspot = float(T.max())
### END SOLUTION
print('hot-spot temperature:', round(hotspot,3))
assert hotspot > 0

hot-spot temperature: 1.466
